# Approach 2

Approach 2 differs from Approach 1, in the sense that the derived reactions from Rhea are used as the foundation for the final dataframe. Reactions will be connected to their relevant proteins in UniProt, before they are mapped to the transporters of TCDB.

Semantically, it will follow something along these lines: R:Data + RID -> UID -> TCID + substrate.

In [2]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd

 By 24.02.2025, Rhea contained 17422 reactions. These lay the foundation for the data set.\
 However, only 1513 of them are transport reactions, according to Rhea. Despite this, all reactions from Rhea are included, in hopes that more transporters will be connected than from transport reactions alone.

In [3]:
rhea = pd.read_csv("../Rhea/Rhea.tsv", sep="\t")
rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
rhea.rename(columns={"Reaction identifier" : "RID"}, inplace=True)

Now, one can obtain the RID-UID mappings.

In [4]:
uid_rid_map = pd.read_csv("../UniProt/Modified_queries/RID_UID.tsv", sep="\t")
uid_rid_map["RID"] = uid_rid_map["RID"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)
df = rhea.merge(uid_rid_map, on="RID", how="left")

Okay, this df is BIG. No need to make it bigger than necessary by attaching RSIDs yet. The first step is to connecting the UIDs possible to TCIDs.\
For this, one needs to obtain the current data from TCDB. And why not just take it all at once, as it is needed later on?

In [5]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def parse_data(uid_txt, substrates_txt, aa_txt, rsid_txt):

    # TCID and Accesion ID (UID/RefSeq)
    uid_data = [line.split("\t") for line in uid_txt.strip().split("\n")]
    df_uid = pd.DataFrame(uid_data, columns=["UID", "TCID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
                        for line in substrates_lines
                        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    # AA sequence (TCID, UID and AA)
    fasta_io = StringIO(aa_txt)
    tc_data = [[record.description.split("|")[3].split()[0],  # TCID
                record.description.split("|")[2],  # UID
                str(record.seq)]  # AA
                for record in SeqIO.parse(fasta_io, "fasta")]
    
    df_aa = pd.DataFrame(tc_data, columns=["TCID", "UID", "AA"])

    # TCID and RSID
    rsid_data = [[line.split("\t")[0], line.split("\t")[1]]
                 for line in rsid_txt.strip().split("\n")]
    df_rsid = pd.DataFrame(rsid_data, columns=["RSID", "TCID"])

    return df_uid, df_substrates, df_aa, df_rsid

In [6]:
tc_uid_url = "https://www.tcdb.org/cgi-bin/projectv/public/acc2tcid.py"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"
tc_aa_url = "https://www.tcdb.org/public/tcdb"
tc_rsid_url = "https://www.tcdb.org/cgi-bin/projectv/public/refseq.py"

tc_uid_txt = fetch_data(tc_uid_url)
tc_substrates_txt = fetch_data(tc_substrates_url)
tc_aa_txt = fetch_data(tc_aa_url)
tc_rsid_txt = fetch_data(tc_rsid_url)

df_uid, df_substrates, df_aa, df_rsid = parse_data(tc_uid_txt, tc_substrates_txt, tc_aa_txt, tc_rsid_txt)

Now merging df with df_uid on the column UID.\
Need to add all AAs with respective UID. Then create a new col, UID2, where all UIDs are converted to RSIDs when possible, and now merge left with df UID2 and df_aa UID for the instances NOT already assigned an AA.

In [7]:
df = df.merge(df_aa, on="UID", how="left")

Mapping UIDs to their RSIDs from UniProt/Modified_queries/UID_RSID.tsv.

In [8]:
uid_rsid_map = pd.read_csv("../UniProt/Modified_queries/UID_RSID.tsv", sep="\t")
df = df.merge(uid_rsid_map, on="UID", how="left")

Now, one can map the RSIDs to the missing TCIDs through df_aa, for the UIDs that actually are RSIDs.

In [9]:
df = df.merge(df_aa.iloc[:,[1,2]], left_on='RSID', right_on='UID', how="left")
df["AA"] = df["AA_x"].fillna(df["AA_y"])
df["UID"] = df["UID_x"].fillna(df["UID_y"])
df.drop(columns=["UID_x", "UID_y", "AA_x", "AA_y"], inplace=True)
df["UID"] = df["UID"].fillna(df["RSID"])
df.drop(columns=["RSID"], inplace=True)
df = df.dropna(subset=["AA"]).reset_index(drop=True)

Now, it is fitting to map the data from TCID to their belonging CHEBI IDs, in order to try to spot any differences! Also converting any secondary CHEBI IDs to primary, if that is the case. The same goes for name conversion.

In [ ]:
df = df.merge(df_substrates, on="TCID", how="left")

df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
df_p2n = pd.read_csv("../ChEBI/p2n.tsv", sep="\t")

secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
primary_to_name = dict(zip(df_p2n["Primary_ID"], df_p2n["ChEBI Name"]))

df["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))
df["CHEBI Name"] = df.apply(lambda row: primary_to_name.get(row["CHEBI ID"], row["CHEBI Name"]), axis=1)

df = df.drop_duplicates()

The purpose of the sections of code below, is to make the df from this approach on the same format, including the same columns with similar names, as the df from A1.
For this, I need to take certain steps:
1) Rename all Rhea-columns (except RID) so that they start with "R:"
2) Extract Family from TCID
3) Include Mechanism, Acting Entity and Reaction from Family

In [12]:
df.rename(columns={'Equation': 'R:Equation', 
                   'ChEBI name': 'R:ChEBI name', 
                   'ChEBI identifier': 'R:ChEBI identifier'}, inplace=True)

In [13]:
# Importing the families and related mechanisms
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_final.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "=", "â†’": "="}, regex=True)

df["Family"] = df["TCID"].apply(lambda x: ".".join(str(x).split(".")[:3]) if pd.notna(x) else x)
df = df.merge(df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left")

In [ ]:
def create_reaction_row(row):

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    

    mechanisms = row["Mechanism"].split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = []
    for mechanism, entity in zip(mechanisms, acting_entities):
        reaction = mechanism.replace(entity, chebi_name)
        reactions.append(reaction)
    
    return ", ".join(reactions)

df["Reaction"] = df.apply(create_reaction_row, axis=1)

The last step is to write it to a .tsv and .fasta.

In [16]:
df_fasta = df.drop_duplicates(subset=["UID", "TCID", "AA"])
fasta_file = "transporters2.fasta"

with open(fasta_file, "w") as f:
    for _, row in df_fasta.iterrows():
        uid = row["UID"]
        tcid = row["TCID"]
        sequence = row["AA"]
        f.write(f">{uid}|{tcid}\n{sequence}\n")

df.to_csv("transporters2_df.tsv", sep="\t", index=False)